# Bayesian M2M Gaussian experiment

This notebook is only the experiment driver and plotting layer.

The computational stages are separate scripts:

1. Generate and freeze \(D=\{(X^i,Y^i)\}_{i=1}^n\) and a separate unseen \((X^0,Y^0)\).
2. Call `collect_pca_trajectory_fixed.py` to pretrain and construct the PCA subspace from this fixed \(D\).
3. Call `run_ess.py` to sample the posterior PCA coordinates \(\phi\).
4. Call `posterior_predictive.py` to evaluate the posterior predictive at \(X^0\).

The notebook produces:
- a pair plot of the \(K\) posterior coordinates \(\phi\);
- the posterior predictive distribution of the target mean \(m_\phi(X^0)\).


In [10]:
from pathlib import Path
import sys
import subprocess
import random

import numpy as np
import torch
import matplotlib.pyplot as plt

# Find project root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "train.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets import GaussianLocationDataset, StudentTLocationDataset
from utils import bayes_optimal_target_mean

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Python:", sys.executable)


Project root: c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example
Device: cpu
Python: c:\Users\alice\miniconda3\envs\m2m_env\python.exe


In [ ]:
# Experiment configuration

SEED = 7

N_TRAIN_MEASURES = 500
N_TEST_MEASURES = 10
N_POINTS = 128

beta = torch.zeros(2)
Sigma_Z = torch.eye(2)
Sigma_X = 0.25 * torch.eye(2)
Sigma_Y = 0.10 * torch.eye(2)
B0 = torch.eye(2)

TRAIN_DF = 5.0

# PCA trajectory
PRETRAIN_EPOCHS = 50
PRETRAIN_LR = 1e-3
TRAJECTORY_EPOCHS = 50
TRAJECTORY_LR = 1e-2
MAX_SNAPSHOTS = 20
PCA_RANK = 5
BATCH_SIZE = 32

# ESS
PRIOR_STD = 1
ESS_BURN_IN = 100
ESS_NUM_SAMPLES = 1000
ESS_THIN = 1
ESS_TEMPERATURE = 1e3
ESS_SEED = 1234

FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [12]:
def save_dataset(dataset, distribution, path):
    source_list = []
    target_list = []
    latent_list = []

    for i in range(len(dataset)):
        X_i, Y_i, Z_i = dataset[i]
        source_list.append(X_i)
        target_list.append(Y_i)
        latent_list.append(Z_i)

    D_source = torch.stack(source_list)
    D_target = torch.stack(target_list)
    D_latent = torch.stack(latent_list)

    fixed_data = {
        "source": D_source,
        "target": D_target,
        "latent": D_latent,
        "dataset_config": {
            "distribution" : distribution,
            "num_measures": len(dataset),
            "beta": beta,
            "Sigma_Z": Sigma_Z,
            "Sigma_X": Sigma_X,
            "Sigma_Y": Sigma_Y,
            "B0": B0,
            "df": TRAIN_DF,
        },
    }

    torch.save(fixed_data, path)

    print("D source:", tuple(D_source.shape))
    print("D target:", tuple(D_target.shape))
    print("Saved:", path)

## 1. Generate and freeze \(D\) and an unseen test pair

We materialize the observations once. The later PCA and ESS scripts use these exact tensors, so the likelihood is deterministic conditional on the fixed dataset.


In [13]:
#generate and save D
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

for dist in ["gaussian", "student_t"]:
    if dist == "gaussian":
        full_train_dataset = GaussianLocationDataset(
            num_measures=N_TRAIN_MEASURES,
            num_samples=N_POINTS,
            beta=beta,
            Sigma_Z=Sigma_Z,
            Sigma_X=Sigma_X,
            Sigma_Y=Sigma_Y,
            B0=B0,
            seed=SEED,
        )
    elif dist == "student_t":
        full_train_dataset = StudentTLocationDataset(
            num_measures=N_TRAIN_MEASURES,
            num_samples=N_POINTS,
            beta=beta,
            Sigma_Z=Sigma_Z,
            Sigma_X=Sigma_X,
            Sigma_Y=Sigma_Y,
            B0=B0,
            df=TRAIN_DF,
            seed=SEED,
        )
    path = (CHECKPOINT_DIR / f"{dist}_fixed_D.pt")
    save_dataset(full_train_dataset, dist, path)




D source: (500, 128, 2)
D target: (500, 128, 2)
Saved: c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_fixed_D.pt
D source: (500, 128, 2)
D target: (500, 128, 2)
Saved: c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\student_t_fixed_D.pt


In [14]:
#generate and save X0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

full_test_dataset = GaussianLocationDataset(
        num_measures=N_TEST_MEASURES,
        num_samples=N_POINTS,
        beta=beta,
        Sigma_Z=Sigma_Z,
        Sigma_X=Sigma_X,
        Sigma_Y=Sigma_Y,
        B0=B0,
        seed=SEED,
    )

path = (CHECKPOINT_DIR / f"gaussian_fixed_test_data.pt")
save_dataset(full_test_dataset, "gaussian", path)

D source: (10, 128, 2)
D target: (10, 128, 2)
Saved: c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_fixed_test_data.pt


## 2. Construct the PCA subspace

The separate collector starts from a fresh transformer, pretrains on the fixed \(D\), continues with constant-step-size SGD, forms the SWA shift \(\hat w\), and computes the scaled PCA basis \(P\), giving

\[
w(\phi)=\hat w+P\phi.
\]


In [15]:
pca_script = PROJECT_ROOT / "collect_pca_trajectory.py"

if not pca_script.exists():
    raise FileNotFoundError(f"Missing {pca_script}")

for dist in ["gaussian", "student_t"]:
    train_data_path = CHECKPOINT_DIR / f"{dist}_fixed_D.pt"
    pca_path = CHECKPOINT_DIR / f"{dist}_pca_subspace.pt"
    
    cmd = [
        sys.executable,
        "-u",
        str(pca_script),
        "--fixed-data", str(train_data_path),
        "--output", str(pca_path),
        "--pretrain-epochs", str(PRETRAIN_EPOCHS),
        "--pretrain-lr", str(PRETRAIN_LR),
        "--trajectory-epochs", str(TRAJECTORY_EPOCHS),
        "--trajectory-lr", str(TRAJECTORY_LR),
        "--max-snapshots", str(MAX_SNAPSHOTS),
        "--pca-rank", str(PCA_RANK),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
    ]

    print("Running PCA collector...", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )

    print(f"{dist} PCA collector finished.", flush=True)


Running PCA collector...
PCA subspace from fixed training dataset D
Fixed dataset : C:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_fixed_D.pt
Output        : C:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_pca_subspace.pt
Device        : cpu
Loading fixed dataset...
source shape: (500, 128, 2)
target shape: (500, 128, 2)
c:\Users\alice\miniconda3\envs\m2m_env\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Trainable parameters: 414,594

Starting pretraining...
[pretrain] epoch 0001/0050 | NLL = 297.921288
[pretrain] epoch 0002/0050 | NLL = 91.966308
[pretrain] epoch 0003/0050 | NLL = 85.793130
[pretrain] epoch 0004/0050 | NLL = 82.423906
[pretrain] epoch 0005/0050 | NLL = 83.582106
[pretrain] epoch 0006/0050 | NLL = 79.1

In [ ]:
##load PCA checkpoint and print info

pca_checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False,
)

print("PCA basis shape:", tuple(pca_checkpoint["pca_basis"].shape))
print("Explained variance ratio:")
print(pca_checkpoint["explained_variance_ratio"])


## 3. Sample \(\phi\mid D\) with elliptical slice sampling

The ESS script targets

\[
p_T(\phi\mid D)
\propto
p(D\mid\phi)^{1/T}p(\phi),
\qquad
\phi\sim N(0,I).
\]

All likelihood evaluations use the frozen dataset \(D\).


In [ ]:
ess_script = PROJECT_ROOT / "run_ess.py"
if not ess_script.exists():
    raise FileNotFoundError(f"Missing {ess_script}")

for dist in ["gaussian", "student_t"]:
    train_data_path = CHECKPOINT_DIR / f"{dist}_fixed_D.pt"
    pca_path = CHECKPOINT_DIR / f"{dist}_pca_subspace.pt"
    posterior_path = CHECKPOINT_DIR / f"{dist}_posterior_samples.pt"
    
    cmd = [
        sys.executable,
        "-u",
        str(ess_script),
        "--fixed-data", str(train_data_path),
        "--pca", str(pca_path),
        "--output", str(posterior_path),
        "--prior-std", str(PRIOR_STD),
        "--burn-in", str(ESS_BURN_IN),
        "--num-samples", str(ESS_NUM_SAMPLES),
        "--thin", str(ESS_THIN),
        "--temperature", str(ESS_TEMPERATURE),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(ESS_SEED),
    ]

    print("Running ESS...", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )


Running ESS...
Starting run_ess.py
Device: cpu
Loading fixed dataset...
Fixed dataset loaded.
Loading PCA checkpoint...
PCA checkpoint loaded.
Source shape: (500, 128, 2)
Target shape: (500, 128, 2)
Constructing subspace model...
c:\Users\alice\miniconda3\envs\m2m_env\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Subspace model constructed.
PCA dimension K = 5
Constructing fixed-data likelihood...
Likelihood constructed.
Elliptical slice sampling
Fixed data          : c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_fixed_D.pt
PCA checkpoint      : c:\Users\alice\OneDrive - Nexus365\Documents\DPhil\Project 2\m2m\simple_gaussian_example\checkpoints\gaussian_pca_subspace.pt
PCA dimension K     : 5
Prior std           : 1.0
Temperature         : 1000.0
Burn-in             : 100
Poster

## 4. Pair plot of the \(K\) posterior PCA coordinates

Diagonal panels show marginal histograms. Off-diagonal panels show posterior scatter plots.


In [ ]:
for dist in ["gaussian", "student_t"]:

    # -------------------------------------------------------------
    # Load posterior samples
    # -------------------------------------------------------------

    posterior_path = (
        CHECKPOINT_DIR
        / f"{dist}_posterior_samples.pt"
    )

    if not posterior_path.exists():
        raise FileNotFoundError(
            f"Missing posterior file: {posterior_path}"
        )

    posterior = torch.load(
        posterior_path,
        map_location="cpu",
        weights_only=False,
    )

    phi_samples = posterior[
        "phi_samples"
    ].numpy()

    K = phi_samples.shape[1]

    # -------------------------------------------------------------
    # Pair plot
    # -------------------------------------------------------------

    fig, axes = plt.subplots(
        K,
        K,
        figsize=(2.1 * K, 2.1 * K),
        squeeze=False,
    )

    for i in range(K):
        for j in range(K):

            ax = axes[i, j]

            if i == j:

                ax.hist(
                    phi_samples[:, i],
                    bins=35,
                    alpha=0.75,
                )

            else:

                ax.scatter(
                    phi_samples[:, j],
                    phi_samples[:, i],
                    s=5,
                    alpha=0.20,
                )

            if i == K - 1:
                ax.set_xlabel(
                    rf"$\phi_{j+1}$"
                )
            else:
                ax.set_xticklabels([])

            if j == 0:
                ax.set_ylabel(
                    rf"$\phi_{i+1}$"
                )
            else:
                ax.set_yticklabels([])

            ax.grid(
                alpha=0.15
            )

    distribution_title = {
        "gaussian": "Gaussian training data",
        "student_t": "Student-t training data",
    }[dist]

    fig.suptitle(
        rf"Posterior samples in PCA coordinates ($K={K}$)"
        f"\n{distribution_title}",
        y=0.995,
    )

    plt.tight_layout()

    # -------------------------------------------------------------
    # Save
    # -------------------------------------------------------------

    figure_path = (
        FIGURE_DIR
        / f"{dist}_phi_pairplot.png"
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    print(
        f"Saved: {figure_path}"
    )

    plt.show()
    plt.close(fig)


## 5. Posterior predictive for the unseen source \(X^0\)

For every posterior sample,

\[
\phi^{(s)}\sim p(\phi\mid D),
\qquad
m_s=m_{\phi^{(s)}}(X^0).
\]

These \(m_s\) are samples from the posterior distribution of the target measure's mean.


In [ ]:
predictive_script = PROJECT_ROOT / "posterior_predictive.py"

if not predictive_script.exists():
    raise FileNotFoundError(
        f"Missing {predictive_script}"
    )

for dist in ["gaussian", "student_t"]:
    pca_path = CHECKPOINT_DIR / f"{dist}_pca_subspace.pt"
    posterior_path = CHECKPOINT_DIR / f"{dist}_posterior_samples.pt"
    test_path = CHECKPOINT_DIR / f"gaussian_fixed_test_data.pt"
    predictive_path = CHECKPOINT_DIR / f"{dist}_posterior_predictive.pt"

    cmd = [
        sys.executable,
        "-u",
        str(predictive_script),
        "--pca", str(pca_path),
        "--posterior", str(posterior_path),
        "--test-data", str(test_path),
        "--output", str(predictive_path),
        "--num-predictive-models",
        str(min(500, ESS_NUM_SAMPLES)),
        "--seed",
        str(ESS_SEED + 1),
    ]

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )

In [ ]:
n_plots = 4
predictive_results = {}

for dist in ["gaussian", "student_t"]:

    predictive_path = (
        CHECKPOINT_DIR
        / f"{dist}_posterior_predictive.pt"
    )

    if not predictive_path.exists():
        raise FileNotFoundError(
            f"Missing {predictive_path}"
        )

    predictive = torch.load(
        predictive_path,
        map_location="cpu",
        weights_only=False,
    )

    predictive_results[dist] = predictive


# ---------------------------------------------------------------------
# Load data needed for the benchmarks
#
# We assume the test data are shared between the two experiments.
# ---------------------------------------------------------------------

test_data = predictive_results["gaussian"]

source_test = (
    test_data["source_test"]
)

latent_test = (
    test_data["latent_test"]
)

# True target population means
true_means = np.stack(
    [
        (
            B0
            @ latent_test[i]
        ).numpy()
        for i in range(n_plots)
    ]
)


# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    n_plots,
    2,
    figsize=(11, 4 * n_plots),
    squeeze=False,
)


# ---------------------------------------------------------------------
# One row per test measure
# ---------------------------------------------------------------------

for i in range(n_plots):

    # ================================================================
    # Collect both distributions first so that they share the same
    # axis limits for this test measure.
    # ================================================================

    samples_gaussian = (
        predictive_results["gaussian"][
            "posterior_means"
        ][:, i, :].numpy()
    )

    samples_student_t = (
        predictive_results["student_t"][
            "posterior_means"
        ][:, i, :].numpy()
    )

    true_mean = (
        B0
        @ latent_test[i]
    ).numpy()

    # Transformer posterior means
    posterior_mean_gaussian = (
        samples_gaussian.mean(axis=0)
    )

    posterior_mean_student_t = (
        samples_student_t.mean(axis=0)
    )

    # ================================================================
    # Shared axis range for the two distributions
    # ================================================================

    all_samples = np.concatenate(
        [
            samples_gaussian,
            samples_student_t,
        ],
        axis=0,
    )

    x = all_samples[:, 0]
    y = all_samples[:, 1]

    x_low, x_high = np.quantile(
        x,
        [0.01, 0.99],
    )

    y_low, y_high = np.quantile(
        y,
        [0.01, 0.99],
    )

    # Make sure the true mean is included.
    x_low = min(
        x_low,
        true_mean[0],
    )

    x_high = max(
        x_high,
        true_mean[0],
    )

    y_low = min(
        y_low,
        true_mean[1],
    )

    y_high = max(
        y_high,
        true_mean[1],
    )

    x_pad = 0.1 * (x_high - x_low)
    y_pad = 0.1 * (y_high - y_low)

    x_min = x_low - x_pad
    x_max = x_high + x_pad

    y_min = y_low - y_pad
    y_max = y_high + y_pad

    plot_range = [
        [x_min, x_max],
        [y_min, y_max],
    ]


    # ================================================================
    # Gaussian training
    # ================================================================

    ax = axes[i, 0]

    counts, x_edges, y_edges = np.histogram2d(
        samples_gaussian[:, 0],
        samples_gaussian[:, 1],
        bins=40,
        range=plot_range,
    )

    probabilities = (
        counts / counts.sum()
    )

    ax.imshow(
        probabilities.T,
        origin="lower",
        extent=[
            x_edges[0],
            x_edges[-1],
            y_edges[0],
            y_edges[-1],
        ],
        aspect="equal",
        interpolation="nearest",
        cmap="viridis",
    )

    ax.scatter(
        posterior_mean_gaussian[0],
        posterior_mean_gaussian[1],
        s=110,
        marker="x",
        linewidths=3,
        color="red",
        label="Transformer posterior mean",
        zorder=10,
    )

    ax.scatter(
        true_mean[0],
        true_mean[1],
        s=170,
        marker="+",
        linewidths=3,
        color="white",
        label=r"True $B_0Z^0$",
        zorder=11,
    )

    ax.set_title(
        f"Gaussian training — test measure {i}"
    )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel(
        "Target mean coordinate 1"
    )

    ax.set_ylabel(
        "Target mean coordinate 2"
    )

    ax.legend(
        loc="best"
    )


    # ================================================================
    # Student-t training
    # ================================================================

    ax = axes[i, 1]

    counts, x_edges, y_edges = np.histogram2d(
        samples_student_t[:, 0],
        samples_student_t[:, 1],
        bins=40,
        range=plot_range,
    )

    probabilities = (
        counts / counts.sum()
    )

    ax.imshow(
        probabilities.T,
        origin="lower",
        extent=[
            x_edges[0],
            x_edges[-1],
            y_edges[0],
            y_edges[-1],
        ],
        aspect="equal",
        interpolation="nearest",
        cmap="viridis",
    )

    ax.scatter(
        posterior_mean_student_t[0],
        posterior_mean_student_t[1],
        s=110,
        marker="x",
        linewidths=3,
        color="red",
        label="Transformer posterior mean",
        zorder=10,
    )

    ax.scatter(
        true_mean[0],
        true_mean[1],
        s=170,
        marker="+",
        linewidths=3,
        color="white",
        label=r"True $B_0Z^0$",
        zorder=11,
    )

    ax.set_title(
        f"Student-t training — test measure {i}"
    )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel(
        "Target mean coordinate 1"
    )

    ax.set_ylabel(
        "Target mean coordinate 2"
    )

    ax.legend(
        loc="best"
    )


# ---------------------------------------------------------------------
# Overall figure title
# ---------------------------------------------------------------------

fig.suptitle(
    r"Posterior predictive distributions of $m_\phi(X^0)$",
    fontsize=16,
    y=0.995,
)

plt.tight_layout(
    rect=[0, 0, 1, 0.985]
)


# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

figure_path = (
    FIGURE_DIR
    / "gaussian_vs_student_t_posterior_predictive.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

print(
    f"Saved: {figure_path}"
)

plt.show()
plt.close(fig)

## Output files

The experiment leaves these reusable files in `checkpoints/`:

- `fixed_D_and_X0.pt` — frozen training dataset and unseen test pair;
- `pca_subspace_fixed_D.pt` — SWA point and PCA subspace;
- `posterior_phi.pt` — ESS samples of \(\phi\);
- `posterior_predictive_X0.pt` — posterior samples of the target mean and one predictive point-cloud realization.
